# Flood Extent Detection — Sentinel-1 SAR (SNAP output → flood map)
Cleaned pipeline: threshold → speckle cleanup → remove permanent water bodies → polygonize → save raster/vector outputs → overlay on a live basemap.

Run cells top to bottom. Only two values normally need tuning: `band_index` (Cell 3) and `threshold` (Cell 5).

In [ ]:
!pip install rasterio geopandas shapely scikit-image contextily osmnx folium --quiet

## 1. Load the SNAP GeoTIFF and inspect it

In [ ]:
import subprocess
import os
import numpy as np
import rasterio
from rasterio.warp import transform_bounds
from rasterio.features import shapes as rio_shapes, rasterize
from rasterio.enums import ColorInterp
from shapely.geometry import box
from scipy.ndimage import binary_dilation
from skimage.morphology import binary_opening, binary_closing, remove_small_objects, remove_small_holes
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

tif_path = "/kaggle/input/datasets/rumble08/kalutra/subset_2_of_S1A_IW_GRDH_1SDV_20240523T002552_20240523T002617_053991_069047_6D9F_Cal_Spk_TC.tif"  # <-- update if your dataset path differs

assert Path(tif_path).exists(), f"File not found: {tif_path}"
print("File found:", tif_path)

In [ ]:
with rasterio.open(tif_path) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Width x Height:", src.width, "x", src.height)
    print("Band count:", src.count)
    print("Dtypes:", src.dtypes)
    print("Nodata:", src.nodata)

    for i in range(1, src.count + 1):
        band = src.read(i).astype(np.float32)
        valid = band[band != 0]
        print(f"\n--- Band {i} ---")
        print("Description:", src.descriptions[i-1])
        print("Non-zero pixels:", valid.size, f"({100*valid.size/band.size:.1f}% of image)")
        if valid.size:
            print("Min / Max / Mean:", valid.min(), valid.max(), valid.mean())

## 2. Convert linear → dB and crop to the real (non-padded) data footprint
`band_index` — set this to whichever band Cell above shows as VV (lower typical backscatter values).

In [ ]:
band_index = 1  # <-- set based on the band stats printed above

with rasterio.open(tif_path) as src:
    linear = src.read(band_index).astype(np.float32)
    profile = src.profile
    transform = src.transform
    crs = src.crs

# treat exact 0.0 as padding, not a real measurement
linear[linear == 0] = np.nan

# linear Sigma0 -> dB (same math as SNAP's Linear-to-dB tool)
dB_full = 10 * np.log10(linear)

# crop to the bounding box of actual valid data (drops the empty padding border)
rows = np.where(~np.all(np.isnan(dB_full), axis=1))[0]
cols = np.where(~np.all(np.isnan(dB_full), axis=0))[0]
r0, r1 = rows.min(), rows.max() + 1
c0, c1 = cols.min(), cols.max() + 1

dB = dB_full[r0:r1, c0:c1]
transform = rasterio.transform.from_origin(
    transform.c + c0 * transform.a,
    transform.f + r0 * transform.e,
    transform.a, -transform.e
)
bounds = rasterio.transform.array_bounds(dB.shape[0], dB.shape[1], transform)

print("Cropped shape:", dB.shape, " (was", dB_full.shape, ")")
print("Valid pixels:", np.sum(~np.isnan(dB)), "/", dB.size)
print("dB min/max/mean:", np.nanmin(dB), np.nanmax(dB), np.nanmean(dB))

plt.figure(figsize=(10, 10))
plt.imshow(dB, cmap="gray")
plt.colorbar(label="Sigma0 dB")
plt.title(f"Sigma0 dB (band {band_index}, cropped)")
plt.show()

## 3. Pick a water threshold

In [ ]:
vals = dB[~np.isnan(dB)].flatten()
plt.hist(vals, bins=150)
plt.axvline(-17, color="red", label="-17 dB")
plt.legend()
plt.title("Backscatter distribution")
plt.show()

## 4. Binary water mask + speckle cleanup

In [ ]:
threshold = -17  # tune based on the histogram above

water_mask = np.where(dB <= threshold, 1, 0).astype(np.uint8)
water_mask[np.isnan(dB)] = 0

plt.imshow(water_mask, cmap="gray")
plt.title(f"Binary mask (threshold = {threshold} dB)")
plt.show()

In [ ]:
mask_bool = water_mask.astype(bool)
mask_clean = binary_opening(mask_bool, footprint=np.ones((3, 3)))
mask_clean = binary_closing(mask_clean, footprint=np.ones((3, 3)))
mask_clean = remove_small_objects(mask_clean, min_size=30)
mask_clean = remove_small_holes(mask_clean, area_threshold=30)

plt.imshow(mask_clean, cmap="gray")
plt.title("Cleaned flood mask")
plt.show()

## 5. Remove permanent water bodies (rivers, lakes, reservoirs) using OSM
This is what gives you *only the flooded portion*, not existing waterbodies.

In [ ]:
minx, miny, maxx, maxy = transform_bounds(crs, "EPSG:4326", *bounds)
aoi_polygon = box(minx, miny, maxx, maxy)
print(aoi_polygon.bounds)

In [ ]:
import osmnx as ox

ox.settings.timeout = 120
bbox = aoi_polygon.bounds  # (minx, miny, maxx, maxy)

# permanent water polygons (lakes, ponds, reservoirs) -- fast, few features
try:
    osm_water_poly = ox.features_from_bbox(bbox=bbox, tags={"natural": "water", "landuse": "reservoir"})
    osm_water_poly = osm_water_poly[osm_water_poly.geometry.type.isin(["Polygon", "MultiPolygon"])]
except Exception as e:
    print("Water polygons failed:", e)
    osm_water_poly = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

# major rivers/canals/streams only -- NOT every drainage line (that's what makes this slow)
try:
    osm_water_lines = ox.features_from_bbox(bbox=bbox, tags={"waterway": ["river", "canal", "stream"]})
except Exception as e:
    print("Waterways failed:", e)
    osm_water_lines = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

osm_water = gpd.GeoDataFrame(
    __import__("pandas").concat([osm_water_poly, osm_water_lines], ignore_index=True), crs="EPSG:4326"
)
print(f"Total permanent water features: {len(osm_water)}")

In [ ]:
if len(osm_water) > 0:
    osm_water_proj = osm_water.to_crs(crs)
    permanent_water = rasterize(
        [(geom, 1) for geom in osm_water_proj.geometry if geom is not None],
        out_shape=dB.shape, transform=transform, fill=0, dtype=np.uint8
    ).astype(bool)
else:
    permanent_water = np.zeros(dB.shape, dtype=bool)

permanent_water_buffered = binary_dilation(permanent_water, iterations=2)
flood_only = mask_clean & (~permanent_water_buffered)

plt.imshow(flood_only, cmap="Blues")
plt.title("Flood mask -- permanent water removed")
plt.show()

## 6. Polygonize → clean vector (drop noise polygons, compute area)

In [ ]:
mask_u8 = flood_only.astype(np.uint8)
print("Nonzero pixels going into polygonize:", int((mask_u8 == 1).sum()))

results = (
    {"properties": {"DN": v}, "geometry": s}
    for s, v in rio_shapes(mask_u8, mask=mask_u8 == 1, transform=transform)
)
flood_gdf = gpd.GeoDataFrame.from_features(list(results), crs=crs)
flood_gdf = flood_gdf[flood_gdf["DN"] == 1].reset_index(drop=True)
print(f"{len(flood_gdf)} flood polygons")

assert len(flood_gdf) > 0, "Empty result -- check the threshold (Cell 5) or OSM water removal (Cell above)"

In [ ]:
# reproject to a local UTM zone so area is in real square meters (not square degrees)
utm_crs = flood_gdf.estimate_utm_crs()
print("Using projected CRS:", utm_crs)

flood_gdf = flood_gdf.to_crs(utm_crs)
flood_gdf["area_m2"] = flood_gdf.geometry.area
flood_gdf = flood_gdf[flood_gdf["area_m2"] > 500].reset_index(drop=True)
print(f"{len(flood_gdf)} polygons remain after removing noise (< 500 m2)")

if len(flood_gdf) == 0:
    print("All polygons were smaller than 500 m2 -- lower the threshold above.")
else:
    flood_dissolved = flood_gdf.dissolve()
    total_km2 = flood_dissolved.geometry.area.iloc[0] / 1e6
    print(f"Total flooded area: {total_km2:.3f} km2")

In [ ]:
flood_gdf.to_file("/kaggle/working/Flood_Extent_Vector.geojson", driver="GeoJSON")
flood_gdf.to_file("/kaggle/working/Flood_Extent_Vector.shp")
print("Saved GeoJSON and Shapefile to /kaggle/working/")

## 7. Rasterize the cleaned vector back to the grid
This keeps the final raster and vector outputs consistent with each other (same noise removal, same 500 m² cleanup).

In [ ]:
flood_gdf_orig_crs = flood_gdf.to_crs(crs)

final_flood_raster = rasterize(
    [(geom, 1) for geom in flood_gdf_orig_crs.geometry if geom is not None],
    out_shape=dB.shape,
    transform=transform,
    fill=0,
    dtype=np.uint8
)
print("Final raster flood pixels:", int(final_flood_raster.sum()))

## 8. Save the final outputs

In [ ]:
# --- Standard binary raster (0/1) for GIS analysis, further processing ---
out_profile = profile.copy()
out_profile.update(
    dtype=rasterio.uint8, count=1, nodata=0,
    height=final_flood_raster.shape[0], width=final_flood_raster.shape[1],
    transform=transform, crs=crs, compress="lzw"
)
with rasterio.open("/kaggle/working/Flood_Final.tif", "w", **out_profile) as dst:
    dst.write(final_flood_raster, 1)
print("Saved: /kaggle/working/Flood_Final.tif")

In [ ]:
# --- QGIS-ready overlay: single band, real NoData value (not 0/alpha-based) ---
# This is the fix for the "black background" problem: QGIS treats NoData
# as transparent automatically, with no dependency on alpha-channel handling.
# 1 = flooded, everything else = NoData (255).
overlay = np.where(final_flood_raster == 1, 1, 255).astype(np.uint8)

overlay_profile = profile.copy()
overlay_profile.update(
    dtype=rasterio.uint8, count=1, nodata=255,
    height=overlay.shape[0], width=overlay.shape[1],
    transform=transform, crs=crs, compress="lzw"
)
with rasterio.open("/kaggle/working/Flood_QGIS_Overlay.tif", "w", **overlay_profile) as dst:
    dst.write(overlay, 1)

print("Saved: /kaggle/working/Flood_QGIS_Overlay.tif")
print("In QGIS: Layer Properties -> Symbology -> Singleband pseudocolor,")
print("set value 1 to red. The rest (255) is registered as NoData and stays transparent.")

In [ ]:
# --- Full-color RGBA version, with alpha explicitly tagged this time ---
# (previously the alpha band existed but was never tagged as alpha,
# so viewers/QGIS rendered the "transparent" pixels as solid black)
rgba = np.zeros((4, final_flood_raster.shape[0], final_flood_raster.shape[1]), dtype=np.uint8)
rgba[0] = np.where(final_flood_raster == 1, 220, 0)   # R
rgba[1] = np.where(final_flood_raster == 1, 20, 0)    # G
rgba[2] = np.where(final_flood_raster == 1, 20, 0)    # B
rgba[3] = np.where(final_flood_raster == 1, 255, 0)   # Alpha

vis_profile = profile.copy()
vis_profile.update(
    dtype=rasterio.uint8, count=4, nodata=None,
    height=final_flood_raster.shape[0], width=final_flood_raster.shape[1],
    transform=transform, crs=crs, compress="lzw", photometric="RGB"
)
with rasterio.open("/kaggle/working/Flood_Visual_RGBA.tif", "w", **vis_profile) as dst:
    dst.write(rgba)
    dst.colorinterp = (ColorInterp.red, ColorInterp.green, ColorInterp.blue, ColorInterp.alpha)

print("Saved: /kaggle/working/Flood_Visual_RGBA.tif")

## 9. Overlay on satellite imagery (static PNG)

In [ ]:
import contextily as cx

flood_web = flood_gdf.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(14, 14))
flood_web.plot(ax=ax, color="red", alpha=0.5, edgecolor="yellow", linewidth=0.8)
cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery, zoom=16)
ax.set_axis_off()
ax.set_title("Flooded Areas Over Satellite Imagery", fontsize=14)

plt.savefig("/kaggle/working/Flood_Overlay_Satellite.png", dpi=250, bbox_inches="tight")
plt.show()

## 10. Interactive HTML map (satellite + street toggle, optional building footprints)

In [ ]:
import folium

flood_4326 = flood_gdf.to_crs(epsg=4326)
centroid = flood_4326.geometry.union_all().centroid

m = folium.Map(location=[centroid.y, centroid.x], zoom_start=15, tiles=None)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics",
    name="Satellite", overlay=False, control=True
).add_to(m)

folium.TileLayer(tiles="OpenStreetMap", name="Street Map", overlay=False, control=True).add_to(m)

folium.GeoJson(
    flood_4326,
    name="Flooded Areas",
    style_function=lambda x: {"fillColor": "red", "color": "yellow", "weight": 1.5, "fillOpacity": 0.45},
    tooltip=folium.GeoJsonTooltip(fields=["area_m2"], aliases=["Area (m2):"])
).add_to(m)

folium.LayerControl().add_to(m)

m.save("/kaggle/working/Flood_Map_Satellite.html")
print("Saved: /kaggle/working/Flood_Map_Satellite.html")
m

In [ ]:
# optional: draw building footprints on top (skips quietly if OSM coverage is sparse)
try:
    buildings = ox.features_from_bbox(bbox=aoi_polygon.bounds, tags={"building": True})
    buildings = buildings[buildings.geometry.type.isin(["Polygon", "MultiPolygon"])]
    print(f"Found {len(buildings)} building footprints")

    folium.GeoJson(
        buildings.to_crs(epsg=4326),
        name="Buildings",
        style_function=lambda x: {"color": "cyan", "weight": 0.5, "fillOpacity": 0}
    ).add_to(m)
    folium.LayerControl().add_to(m)
    m.save("/kaggle/working/Flood_Map_Satellite.html")
except Exception as e:
    print("Building footprints failed (OSM coverage may be sparse here):", e)
m